# Zeroth-Order Optimization: MeZO, Synthetic Gradients (DNI), and HSIC Bottleneck

### Historical Lineage & Theoretical Foundations
- **MeZO** (*Malladi et al., 2023; Spall, 1992 SPSA*): Solves the activation memory wall by executing zeroth-order parameter updates via dual forward passes, regenerating perturbations on-the-fly from a 1-integer RNG seed with inference-equivalent memory footprint.
- **Decoupled Neural Interfaces (DNI)** (*Jaderberg et al., 2017*): Eliminates backward locking by training auxiliary synthetic models to predict error gradients during the forward pass, updating parameters asynchronously.
- **HSIC Bottleneck** (*Ma et al., 2020*): Non-parametric layer-wise training optimizing Hilbert-Schmidt Independence Criterion surrogates $\max \text{HSIC}(H, Y) - \beta \text{HSIC}(H, X)$ without global backprop.

> [!WARNING]
> **Decision Boundary (When NOT to Train with MeZO):** Never attempt scratch training from random initialization using MeZO; zeroth-order gradient variance scales with parameter dimension $d$, making early optimization intractable. Restrict MeZO strictly to fine-tuning pre-trained checkpoints in smooth low-rank loss basins or black-box non-differentiable pipelines.

**Topic Connections:** `🔗 Upstream`: [Non-Backprop & Forward-Forward](05_non_backprop_forward_forward_predictive_coding_eqprop_dfa_hebbian.ipynb) | `🔗 Downstream`: [Scaling Laws, MinHash & Model Collapse](07_scaling_laws_chinchilla_data_dedup_doremi_and_model_collapse.ipynb)


## 2. Mathematical Foundations & Governing Equations

1. **MeZO Directional SPSA Derivative (Malladi et al., 2023):** With $z \sim \mathcal{N}(0, I_d)$:
$$\hat{d} = \frac{\mathcal{L}(\theta + \epsilon z) - \mathcal{L}(\theta - \epsilon z)}{2\epsilon}, \quad \theta_{t+1} = \theta_t - \eta \hat{d} z$$
2. **Decoupled Neural Interfaces (Jaderberg et al., 2017):**
$$\hat{\delta}_l = h_l W_\text{synth} + b_\text{synth}, \quad \mathcal{L}_\text{DNI} = \frac{1}{2B} \sum_{i=1}^B \|\hat{\delta}_{l,i} - \delta^*_{l,i}\|^2_2$$
3. **HSIC Linear Kernel Bottleneck (Ma et al., 2020):** With $H_c = I_B - \frac{1}{B} \mathbf{1}\mathbf{1}^T$:
$$K = H H^T, \quad L = Y Y^T, \quad \text{HSIC}(H, Y) = \frac{1}{(B-1)^2} \text{Tr}\big(H_c K H_c H_c L H_c\big)$$
4. **Asymptotic Complexity:** MeZO $2 \times \text{FLOP}_\text{fwd}$, $\mathcal{O}(1)$ activation memory; DNI $\mathcal{O}(B D^2)$; HSIC $\mathcal{O}(B^2 D + B^3)$.


In [ ]:
import sys
from pathlib import Path
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F

repo_root = Path.cwd().resolve()
while not (repo_root / "core").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
import core
core.setup_env(seed=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt_path = repo_root / "weights/04_training_methods_and_optimisers/03_training_dynamics_self_optimising_non_backprop/06_zeroth_order_mezo_synthetic_gradients_and_hsic_bottleneck.safetensors"
weights, test_in, test_out, meta = core.parity.load_reference_checkpoint(ckpt_path, device=device)
print(f"Device: {device} | Checkpoint: {ckpt_path.name} | Loaded {len(weights)} tensors")


In [ ]:
class LinearAnchor(nn.Module):
    def __init__(self, in_features: int = 8, out_features: int = 4):
        super().__init__()
        self.fc = nn.Linear(in_features, out_features)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

def mezo_step(theta: torch.Tensor, z: torch.Tensor, theta_star: torch.Tensor, eps: float = 1e-3, eta: float = 0.05) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    def loss_fn(p: torch.Tensor) -> torch.Tensor:
        return 0.5 * torch.sum((p - theta_star) ** 2)
    lp = loss_fn(theta + eps * z)
    ln = loss_fn(theta - eps * z)
    dir_deriv = (lp - ln) / (2.0 * eps)
    return lp, ln, dir_deriv, theta - eta * dir_deriv * z

def dni_synthetic_gradient(h: torch.Tensor, w_synth: torch.Tensor, b_synth: torch.Tensor, target_grad: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    pred_grad = torch.matmul(h, w_synth) + b_synth
    return pred_grad, 0.5 * torch.mean((pred_grad - target_grad) ** 2)

def hsic_bottleneck(h: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    b = h.shape[0]
    h_c = torch.eye(b) - torch.ones(b, b) / float(b)
    k_c = h_c @ (h @ h.T) @ h_c
    l_c = h_c @ (y @ y.T) @ h_c
    return (1.0 / ((b - 1.0) ** 2)) * torch.trace(k_c @ l_c)
print("Zeroth-order & decoupled interfaces defined successfully.")


In [ ]:
anchor = LinearAnchor(8, 4).to(device)
anchor.fc.weight.data.copy_(weights['anchor_w'].T)
anchor.fc.bias.data.copy_(weights['anchor_b'])

lp, ln, dd, th_next = mezo_step(weights['theta'], weights['z'], weights['theta_star'], eps=1e-3, eta=0.05)
err_mezo = max(float(torch.abs(lp - weights['golden_loss_pos'][0]).item()),
               float(torch.abs(ln - weights['golden_loss_neg'][0]).item()),
               float(torch.abs(dd - weights['golden_dir_deriv'][0]).item()),
               float(torch.max(torch.abs(th_next - weights['golden_theta_next'])).item()))

pred_g, dni_loss = dni_synthetic_gradient(weights['h'], weights['w_synth'], weights['b_synth'], weights['target_grad'])
err_dni = max(float(torch.max(torch.abs(pred_g - weights['golden_pred_grad'])).item()),
              float(torch.abs(dni_loss - weights['golden_dni_loss'][0]).item()))

hsic_score = hsic_bottleneck(weights['hsic_h'], weights['hsic_y'])
err_hsic = float(torch.abs(hsic_score - weights['golden_hsic_score'][0]).item())

anchor_out = anchor(weights['anchor_x'])
err_anchor = float(torch.max(torch.abs(anchor_out - weights['golden_anchor_out'])).item())

for name, err in [('MeZO Step', err_mezo), ('DNI Gradient', err_dni), ('HSIC Score', err_hsic), ('Anchor', err_anchor)]:
    assert err < 1e-5, f'{name} invariant failed: {err:.2e}'
print(f'All 4 invariants verified! MeZO: {err_mezo:.2e}, DNI: {err_dni:.2e}, HSIC: {err_hsic:.2e}, Anchor: {err_anchor:.2e}')


In [ ]:
track_fns = {
    'Linear Anchor': lambda: anchor(weights['anchor_x']),
    'MeZO Forward Step': lambda: mezo_step(weights['theta'], weights['z'], weights['theta_star']),
    'DNI Synthetic Prediction': lambda: dni_synthetic_gradient(weights['h'], weights['w_synth'], weights['b_synth'], weights['target_grad']),
    'HSIC Kernel Trace': lambda: hsic_bottleneck(weights['hsic_h'], weights['hsic_y']),
}
bench_results = core.benchmark.compare_tracks(track_fns, warmup=3, iters=15, batch_size=4)
print(core.benchmark.format_benchmark_table(bench_results))


In [ ]:
# Pathological Anti-Pattern: Materializing perturbation vector z in RAM vs in-place RNG re-seeding
# Simulating memory allocation overhead of materializing z across 10 million parameters
num_params = 10_000_000
bytes_materialized = num_params * 4 # 40 MB for single parameter buffer
bytes_mezo_seed = 8 # Single 64-bit integer seed
memory_savings_ratio = bytes_materialized / bytes_mezo_seed

# Verification of RNG re-seeding determinism
seed = 12345
torch.manual_seed(seed)
z_step1 = torch.randn(8)
torch.manual_seed(seed) # Re-seeding reproduces identical perturbation on-the-fly
z_step2 = torch.randn(8)
seed_parity_err = float(torch.max(torch.abs(z_step1 - z_step2)).item())

print(f'[FAILURE DEMONSTRATED] Storing explicit perturbation vectors consumes {bytes_materialized / (1024*1024):.1f} MB!')
print(f'[CORRECT] In-place RNG re-seeding uses {bytes_mezo_seed} bytes ({memory_savings_ratio:,.0f}x memory saving, parity error: {seed_parity_err:.1e})!')
assert seed_parity_err == 0.0 and memory_savings_ratio > 1_000_000, 'MeZO RNG re-seeding assertion failed.'
print('[REMEDY] Regenerate perturbation z dynamically from an integer seed; never materialize or store full parameter-sized noise tensors in RAM.')


## 8. Algorithmic Takeaways & Cross-Domain Isomorphisms

- **Takeaways**: MeZO fine-tunes deep neural networks in the exact memory footprint of inference via seed-regenerated SPSA; DNI breaks backward locking via auxiliary synthetic gradient predictors; HSIC enables non-parametric layer-wise representation discovery.

| Scientific Domain | MeZO Zeroth-Order SPSA | DNI Synthetic Gradients | HSIC Bottleneck |
| :--- | :--- | :--- | :--- |
| **Large Language Models** | Memory-constrained fine-tuning on consumer GPUs | Pipeline parallel layer decoupled updates | Feature selection without downstream task loss |
| **Quantum Parameter Tuning** | Parameter-shift gradient approximation on physical QPUs | Asynchronous quantum-classical interface updates | Quantum state fidelity mutual information maximization |
| **Analog Neuromorphic Chips** | Dual-pulse voltage perturbation on memristors | Local error estimation on distributed neural cores | Kernel cross-correlation tuning on sensor streams |

---
**Navigation:** [← Non-Backprop & Forward-Forward](05_non_backprop_forward_forward_predictive_coding_eqprop_dfa_hebbian.ipynb) | [↑ Master Index](../../README.md) | [Scaling Laws, MinHash & Model Collapse →](07_scaling_laws_chinchilla_data_dedup_doremi_and_model_collapse.ipynb)
